In [ ]:
# !python --version

# !pip install git+https://github.com/IBM/terratorch.git

# # # In a Kaggle notebook cell, run this first:
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 --force-reinstall

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import sys 
import os
import sys
import numpy as np
import torch
import glob

import terratorch
from terratorch.datamodules import MultiTemporalCropClassificationDataModule
from terratorch.tasks import SemanticSegmentationTask
from terratorch.datasets.transforms import FlattenTemporalIntoChannels, UnflattenTemporalFromChannels

import albumentations
from albumentations.pytorch import ToTensorV2

import lightning.pytorch as pl
from lightning.pytorch.loggers import TensorBoardLogger
from lightning.pytorch.callbacks import ModelCheckpoint
import rasterio
from pathlib import Path

In [ ]:
# Change path for kaggle
dataset_path = '/kaggle/input/datasets/manipes/ibm-flood-dataset-phase-2/data/'

In [ ]:
datamodule = terratorch.datamodules.GenericNonGeoSegmentationDataModule(
    batch_size=2,
    num_workers=2,
    num_classes=3,

    # Define dataset paths
    train_data_root=dataset_path+'image',
    train_label_data_root=dataset_path+'label',
    val_data_root=dataset_path+'image',
    val_label_data_root=dataset_path+'label',
    test_data_root=dataset_path+'image',
    test_label_data_root=dataset_path+'label',

    # Define splits
    train_split=dataset_path+'split/train.txt',
    val_split=dataset_path+'split/val.txt',
    test_split=dataset_path+'split/test.txt',

    img_grep='*image.tif',
    label_grep='*label.tif',

    train_transform=[
        albumentations.D4(), # Random flips and rotation
        
        albumentations.ShiftScaleRotate(
            shift_limit=0.1,
            scale_limit=0.15,
            rotate_limit=30,                    
            border_mode=0,
            p=0.5
        ),

        albumentations.RandomBrightnessContrast(
            brightness_limit=0.2,
            contrast_limit=0.2,
            p=0.5
        ),
        albumentations.GaussNoise(
            std_range=(0.02, 0.08),
            p=0.3
        ),


        albumentations.CoarseDropout(
            num_holes_range=(4, 8),
            hole_height_range=(32, 64),
            hole_width_range=(32, 64),
            fill=0,                                   # fill with no-data value
            p=0.3
        ),
        albumentations.pytorch.transforms.ToTensorV2()
    ],
    val_transform=None,  # Using ToTensor() by default
    test_transform=None,

    # dataset_bands = [1,2,3,4,5,6],
    # output_bands = [1,2],
    
    # mean and stds of HH, HV, Band1 (Green), Band2 (Red), Band3 (NIR), Band4 (SWIR)
    means = [841.1257, 371.6175 ,1734.1789, 1588.3142, 1742.8452, 1218.5551],
    stds = [473.7090, 170.3611, 623.0474, 612.8465, 564.5835, 528.0894],
    no_data_replace=0,
    no_label_replace=-1,
    # We use all six bands of the data, so we don't need to define dataset_bands and output_bands.
    # Change path for kaggle
    predict_data_root = "/kaggle/input/datasets/manipes/ibm-flood-dataset-phase-2/data/prediction/image/"
)

# Setup train and val datasets
datamodule.setup("fit")

In [ ]:
# checking datasets train split size
train_dataset = datamodule.train_dataset
print(len(train_dataset))

# checking datasets validation split size
val_dataset = datamodule.val_dataset
len(val_dataset)



In [ ]:
# # plotting a few samples
# val_dataset.plot(val_dataset[0])
# val_dataset.plot(val_dataset[1])

In [ ]:
label_dir   = Path(dataset_path) / "label"
train_split = Path(dataset_path) / "split" / "train.txt"

with open(train_split) as f:
    train_files = [line.strip() for line in f.readlines()]

print(f"Total training patches: {len(train_files)}")
print("-" * 40)

total_pixels   = 0
total_flood    = 0
total_no_flood = 0
total_wb = 0
total_ignore   = 0

for filename in train_files:
    label_file = label_dir / f"{filename}_label.tif"

    with rasterio.open(label_file) as src:
        mask = src.read(1)   # [H, W]

    flood    = (mask == 1).sum()
    no_flood = (mask == 0).sum()
    water_body = (mask == 2).sum()
    ignore   = (mask == -1).sum()
    total    = mask.size

    total_pixels   += total
    total_flood    += flood
    total_no_flood += no_flood
    total_ignore   += ignore
    total_wb += water_body

    print(f"{filename}")
    print(f"  Flood:    {flood:>8,}  ({100*flood/total:.1f}%)")
    print(f"  No-Flood: {no_flood:>8,}  ({100*no_flood/total:.1f}%)")
    print(f"  Water Body: {water_body:>8,}  ({100*water_body/total:.1f}%)")
    print(f"  Ignore:   {ignore:>8,}  ({100*ignore/total:.1f}%)")

valid_pixels = total_flood + total_no_flood + total_wb
print("\n" + "=" * 40)
print("OVERALL SUMMARY")
print("=" * 40)
print(f"Total patches:       {len(train_files)}")
print(f"Total pixels:        {total_pixels:,}")
print(f"Valid pixels:        {valid_pixels:,}")
print(f"Flood pixels:        {total_flood:,}  ({100*total_flood/valid_pixels:.1f}%)")
print(f"No-Flood pixels:     {total_no_flood:,}  ({100*total_no_flood/valid_pixels:.1f}%)")
print(f"Water body pixels:   {total_wb:,}  ({100*total_wb/valid_pixels:.1f}%)")
print(f"Ignored pixels:      {total_ignore:,}  ({100*total_ignore/total_pixels:.1f}%)")


In [ ]:
# Class Weight Computation 
n_samples  = total_flood + total_no_flood + total_wb 
n_classes  = 3

w_no_flood   = n_samples / (n_classes * total_no_flood)  # class 0
w_flood      = n_samples / (n_classes * total_flood)     # class 1
w_water_body = n_samples / (n_classes * total_wb)        # class 2

class_weights = [w_no_flood, w_flood, w_water_body]

print(f"No-Flood   (class 0): {w_no_flood:.4f}")
print(f"Flood      (class 1): {w_flood:.4f}")
print(f"Water Body (class 2): {w_water_body:.4f}")
print(f"\nCLASS_WEIGHTS = {[round(w, 2) for w in class_weights]}")

In [ ]:
#Hyperparameters
EPOCHS = 100 # Set number of epochs; 1 for testing
BATCH_SIZE = 4 
LR = 1.0e-4
WEIGHT_DECAY = 0.05
HEAD_DROPOUT=0.2
FREEZE_BACKBONE = True

BANDS =[1,2,3,4,5,6]
NUM_FRAMES = 1
CLASS_WEIGHTS =[0.51,2.19,1.71]
SEED = 0
OUT_DIR='/kaggle/working/'


In [ ]:
from lightning.pytorch.callbacks import EarlyStopping
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from typing import Any

SEED = 0

pl.seed_everything(SEED)

# Logger
logger = TensorBoardLogger(
    save_dir=OUT_DIR,
    name="test",
)

# Callbacks
checkpoint_callback = ModelCheckpoint(
    monitor="val/mIoU",
    mode="max",
    dirpath=os.path.join(OUT_DIR, "test", "checkpoints"),
    filename="best-checkpoint-{epoch:02d}-{val_loss:.2f}",
    save_top_k=1,
)

early_stopping = EarlyStopping(
    monitor="val/mIoU",
    mode="max",
    patience=50,
    verbose=True,       
)

# Trainer
trainer = pl.Trainer(
    accelerator="auto",
    strategy="auto",
    devices="auto",
    precision="bf16-mixed",
    num_nodes=1,
    logger=logger,
    max_epochs=EPOCHS,
    check_val_every_n_epoch=1,
    log_every_n_steps=10,
    enable_checkpointing=True,
    callbacks=[checkpoint_callback],
    # limit_predict_batches=1,  # predict only in the first batch for generating plots
)

# DataModule
data_module = datamodule


# Model

decoder_args = dict(
    decoder="UperNetDecoder",
    decoder_channels=128,
    decoder_scale_modules=True,
)

necks = [
    dict(
            name="SelectIndices",
            indices=[2, 5, 8, 11]    # indices for prithvi_vit_100
           # indices=[5, 11, 17, 23],   # indices for prithvi_eo_v2_300
    #        # indices=[7, 15, 23, 31]  # indices for prithvi_eo_v2_600
        ),
    dict(
            name="ReshapeTokensToImage",
            effective_time_dim=NUM_FRAMES,
        ),
    dict(name="LearnedInterpolateToPyramidal"),
    ]

backbone_args = dict(
    backbone_pretrained=True,
    backbone="prithvi_eo_v2_tiny_tl", # other models are availble like prithvi_eo_v2_300, prithvi_eo_v2_tiny_tl, prithvi_eo_v2_600, prithvi_eo_v2_600_tl
    #backbone_coords_encoding=["time", "location"],
    backbone_bands=BANDS,
    backbone_num_frames=1, # 1 is the default value,    
    # backbone_pretrained_cfg_overlay=None
    )

model_args = dict(
    **backbone_args,
    **decoder_args,
    num_classes=3,
    head_dropout=HEAD_DROPOUT,
    necks=necks,
    rescale=True,
)
    

# model = SemanticSegmentationTask(
#     model_args=model_args,
#     plot_on_val=False,
#     class_weights=CLASS_WEIGHTS,
#     loss="ce",
#     lr=LR,
#     optimizer="AdamW",
#     optimizer_hparams=dict(weight_decay=WEIGHT_DECAY),
#     ignore_index=-1,
#     freeze_backbone=FREEZE_BACKBONE,
#     freeze_decoder=False,
#     model_factory="EncoderDecoderFactory",
#     scheduler="ReduceLROnPlateau",      
#     scheduler_hparams={                    
#         "mode":     "max",                  
#         "factor":   0.5,                    
#         "patience": 20,     
#         "min_lr":   1e-8,                
#         },
#     # tiled_inference_on_validation=True,
#     # tiled_inference_on_testing=True,
#     # tiled_inference_parameters={
#     #     "h_crop":    256,    # tile height
#     #     "w_crop":    256,    # tile width
#     #     "h_stride":  192,    # stride height (overlap = crop - stride = 64px)
#     #     "w_stride":  192,    # stride width
#     #     "batch_size": 4,     # tiles processed at once
#     # }
# )



# ── Focal Loss ────────────────────────────────────────────────────────────────

class FocalLoss(nn.Module):
    def __init__(self, class_weights, ignore_index=-1, gamma=2.0):
        super().__init__()
        self.ignore_index = ignore_index
        self.gamma        = gamma
        self.register_buffer("class_weights", torch.tensor(class_weights, dtype=torch.float32))

    def forward(self, logits: Tensor, targets: Tensor) -> Tensor:
        # Standard weighted CE first
        ce = F.cross_entropy(
            logits,
            targets,
            weight=self.class_weights.to(logits.device),
            ignore_index=self.ignore_index,
            reduction="none",
        )
        # Get probability of true class for focal modulation
        probs       = F.softmax(logits, dim=1)
        valid_mask  = targets != self.ignore_index
        safe_targets = targets.clone()
        safe_targets[~valid_mask] = 0

        pt          = probs.gather(1, safe_targets.unsqueeze(1)).squeeze(1)
        focal_weight = (1 - pt) ** self.gamma

        loss = (focal_weight * ce)
        return loss[valid_mask].mean()


# ── Multiclass Dice Loss ──────────────────────────────────────────────────────

class MulticlassDiceLoss(nn.Module):
    def __init__(self, ignore_index=-1, smooth=1e-6):
        super().__init__()
        self.ignore_index = ignore_index
        self.smooth       = smooth

    def forward(self, logits: Tensor, targets: Tensor) -> Tensor:
        n_classes   = logits.shape[1]
        probs       = F.softmax(logits, dim=1)           # [B, C, H, W]

        valid_mask  = (targets != self.ignore_index)     # [B, H, W]
        safe_targets = targets.clone()
        safe_targets[~valid_mask] = 0

        one_hot = F.one_hot(safe_targets, n_classes)     # [B, H, W, C]
        one_hot = one_hot.permute(0, 3, 1, 2).float()   # [B, C, H, W]

        # Zero out ignored pixels
        one_hot *= valid_mask.unsqueeze(1).float()
        probs   *= valid_mask.unsqueeze(1).float()

        dims         = (0, 2, 3)                         # reduce over B, H, W
        intersection = (probs * one_hot).sum(dim=dims)
        cardinality  = (probs + one_hot).sum(dim=dims)
        dice_per_class = 1 - (2 * intersection + self.smooth) / (cardinality + self.smooth)

        return dice_per_class.mean()


# ── Composite Loss ────────────────────────────────────────────────────────────

class FocalDiceLoss(nn.Module):
    def __init__(
        self,
        class_weights,
        ignore_index = -1,
        gamma        = 2.0,
        focal_weight = 0.5,
        dice_weight  = 0.5,
    ):
        super().__init__()
        self.focal_weight = focal_weight
        self.dice_weight  = dice_weight
        self.focal        = FocalLoss(class_weights, ignore_index, gamma)
        self.dice         = MulticlassDiceLoss(ignore_index)

    def forward(self, logits: Tensor, targets: Tensor) -> Tensor:
        return (
            self.focal_weight * self.focal(logits, targets) +
            self.dice_weight  * self.dice(logits, targets)
        )


# ── Subclassed Task ───────────────────────────────────────────────────────────

class FocalDiceSegmentationTask(SemanticSegmentationTask):
    def __init__(
        self,
        *args: Any,
        gamma       : float = 2.0,
        focal_weight: float = 0.5,
        dice_weight : float = 0.5,
        **kwargs: Any,
    ) -> None:
        self.gamma        = gamma
        self.focal_weight = focal_weight
        self.dice_weight  = dice_weight
        super().__init__(*args, **kwargs)

    def configure_losses(self) -> None:
        self.criterion = FocalDiceLoss(
            class_weights = self.hparams.get("class_weights", None),
            ignore_index  = self.hparams.get("ignore_index", -1),
            gamma         = self.gamma,
            focal_weight  = self.focal_weight,
            dice_weight   = self.dice_weight,
        )


# ── Model ─────────────────────────────────────────────────────────────────────

model = FocalDiceSegmentationTask(
    model_args=model_args,
    plot_on_val=False,
    class_weights=CLASS_WEIGHTS,
    loss="ce",                   # required by parent, overridden internally
    lr=LR,
    optimizer="AdamW",
    optimizer_hparams=dict(weight_decay=WEIGHT_DECAY),
    ignore_index=-1,
    freeze_backbone=FREEZE_BACKBONE,
    freeze_decoder=False,
    model_factory="EncoderDecoderFactory",
    scheduler="ReduceLROnPlateau",
    scheduler_hparams={
        "mode":     "max",
        "factor":   0.5,
        "patience": 10,
        "min_lr":   1e-8,
    },
    tiled_inference_on_validation=True,
    tiled_inference_on_testing=True,
    tiled_inference_parameters={
        "h_crop":    256,
        "w_crop":    256,
        "h_stride":  192,
        "w_stride":  192,
        "batch_size": 2,
    },
    # Composite loss parameters
    gamma       =2.0,   # focal modulation — higher = more focus on hard pixels
    focal_weight=0.5,   # increase to 0.6 if flood recall is low
    dice_weight =0.5,   # increase to 0.6 if flood IoU is low
)

In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()
trainer.fit(model, datamodule=data_module)

In [ ]:
# !ls output/test/checkpoints

datamodule.setup("test")
#sample checkpoint from 37th epoch for testing..
# best_ckpt_path = "output/test/checkpoints/best-checkpoint-epoch=194-val_loss=0.00.ckpt"

In [ ]:
checkpoint_dir = os.path.join(OUT_DIR, "test", "checkpoints")

all_ckpts = glob.glob(os.path.join(checkpoint_dir, "*.ckpt"))
for ckpt in sorted(all_ckpts):
    print(" ", os.path.basename(ckpt))

best_ckpt_path = checkpoint_callback.best_model_path

In [ ]:
# test_results = trainer.test(model, datamodule=datamodule, ckpt_path=best_ckpt_path)

checkpoint = torch.load(best_ckpt_path, map_location="cuda")
model.load_state_dict(checkpoint["state_dict"], strict=False)

val_results = trainer.validate(
    model,
    datamodule=datamodule,
)
print(val_results)

In [ ]:
datamodule.setup("predict")

predictions = trainer.predict(
    model,
    datamodule=datamodule,
    ckpt_path=best_ckpt_path
)
# change path for kaggle
output_dir = "/kaggle/working/prediction"
os.makedirs(output_dir, exist_ok=True)

for batch_idx, (preds, file_paths) in enumerate(predictions):
    
    if isinstance(preds, tuple):
        preds = preds[0]

    if preds.ndim == 4:               # [B, C, H, W]
        preds = preds.argmax(dim=1)   # [B, H, W]

    preds = preds.cpu().numpy().astype("int16")

    for i in range(preds.shape[0]):
        arr = preds[i]
        arr[arr < 0] = -1             #  enforce ignore_index

        ref_path = file_paths[i]
        with rasterio.open(ref_path) as src:
            meta = src.meta.copy()

        meta.update({
            "count": 1,
            "dtype": "int16",
            "nodata": -1,
            "compress": "lzw",
        })

        out_name = (
            os.path.splitext(os.path.basename(ref_path))[0]
            + ".tif"
        )
        out_path = os.path.join(output_dir, out_name)

        with rasterio.open(out_path, "w", **meta) as dst:
            dst.write(arr, 1)

        print(f"Saved {out_path}")


In [ ]:
# #def mask_to_rle(mask):
#     """
#     Convert binary mask to RLE (Kaggle format).
#     Mask must be 2D numpy array with values 0 or 1.
#     """
#     pixels = mask.flatten(order="F")  # column-major
#     pixels = np.concatenate([[0], pixels, [0]])
#     runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
#     runs[1::2] -= runs[::2]
#     return " ".join(str(x) for x in runs)

In [ ]:
import pandas as pd

def mask_to_rle(mask: np.ndarray) -> str:
    if mask.sum() == 0:
        return '0 0'
    pixels = mask.flatten(order='F')
    pixels = np.concatenate([[0], pixels, [0]])
    runs   = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs)


def rle_to_mask(rle_str: str, shape) -> np.ndarray:
    mask    = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    rle_str = str(rle_str).strip()
    if rle_str in ('0 0', '', '-1', 'nan'):
        return mask.reshape(shape, order='F')
    parts = list(map(int, rle_str.split()))
    for start, length in zip(parts[0::2], parts[1::2]):
        mask[start - 1: start - 1 + length] = 1
    return mask.reshape(shape, order='F')


def generate_submission_csv(pred_dir: str, output_csv: str) -> pd.DataFrame | None:
    pred_dir  = Path(pred_dir)
    tif_files = sorted(pred_dir.glob('*.tif'))

    if len(tif_files) == 0:
        print(f'No prediction TIFs found in {pred_dir}')
        return None

    rows = []

    for tif_path in tif_files:
        with rasterio.open(tif_path) as src:
            pred = src.read(1)

        flood_mask = (pred == 1).astype(np.uint8)
        rle        = mask_to_rle(flood_mask)

        if len(rows) == 0:
            decoded = rle_to_mask(rle, flood_mask.shape)
            status  = "PASS" if np.array_equal(flood_mask, decoded) else "FAIL"
            print(f'RLE round-trip check: {status}')

        # Strip _image.tif to match competition id format
        image_id = tif_path.name.replace('_image.tif', '')
        rows.append({'id': image_id, 'rle_mask': rle})

    df = pd.DataFrame(rows, columns=['id', 'rle_mask'])

    # Ensure no nulls or blanks — competition requires '0 0' for empty masks
    df['rle_mask'] = df['rle_mask'].replace('', '0 0').fillna('0 0')

    df.to_csv(output_csv, index=False)

    n_flood    = (df['rle_mask'] != '0 0').sum()
    n_no_flood = (df['rle_mask'] == '0 0').sum()
    print(f'\nSubmission CSV saved  : {output_csv}')
    print(f'Total patches         : {len(df)}')
    print(f'With flood pixels     : {n_flood}  ({100*n_flood/len(df):.1f}%)')
    print(f'No flood ("0 0")      : {n_no_flood}  ({100*n_no_flood/len(df):.1f}%)')
    print(df.head(5).to_string(index=False))
    return df


submission_csv = str(Path(OUT_DIR) / 'submission.csv')
df_sub = generate_submission_csv(output_dir, submission_csv)

In [ ]:
# import rasterio
# import pandas as pd
# from pathlib import Path
# # change path for kaggle
# tif_dir = Path("/kaggle/working/prediction")   # folder with .tif files
# output_csv = "/kaggle/working/prediction/submission.csv"

# rows = []

# for tif_path in sorted(tif_dir.glob("*.tif")):
#     with rasterio.open(tif_path) as src:
#         mask = src.read(1)

#     # Convert to binary
#     mask = (mask == 1).astype(np.uint8)

#     rle = mask_to_rle(mask)

#     rows.append({
#         "id": tif_path.name.replace("_image.tif", ""), 
#         "rle_mask": rle
#     })

# df = pd.DataFrame(rows)
# df = df.replace("", 0).fillna(0) #replace null/ na with zero - kaggle compatible
# df.to_csv(output_csv, index=False)
# print(f"Saved Kaggle RLE CSV : {output_csv}")
